# 📊 Week 2 Task: Data Preprocessing & Feature Engineering
## Yuva Internship — Artificial Intelligence Trainee
### Project: Predicting Student Performance Using Machine Learning

---

**Author:** AI Trainee — Yuva Internship  
**Dataset:** UCI Student Performance Dataset (Cortez & Silva, 2008)  
**Objective:** Design and document a comprehensive data preprocessing and feature engineering pipeline to prepare the dataset for machine learning model training.

---

## 📌 Notebook Structure

| # | Section |
|---|---------|
| 1 | Environment Setup & Library Imports |
| 2 | Dataset Loading & Initial Exploration |
| 3 | Data Cleaning & Missing Value Treatment |
| 4 | Exploratory Data Analysis (EDA) with Visualizations |
| 5 | Outlier Detection & Treatment |
| 6 | Encoding Categorical Variables |
| 7 | Feature Scaling & Normalization |
| 8 | Feature Engineering |
| 9 | Feature Selection & Correlation Analysis |
| 10 | Final Pipeline & Reproducibility |
| 11 | Summary & Conclusion |

> **How to Run:** Execute each cell top-to-bottom using `Shift+Enter`. No external files required — the dataset is downloaded automatically from the UCI repository.


---
## 1. 🔧 Environment Setup & Library Imports

We begin by importing all required libraries. Each library serves a specific role in the pipeline:
- **pandas / numpy** — data manipulation and numerical operations
- **matplotlib / seaborn** — static visualizations
- **sklearn** — preprocessing utilities and pipeline construction
- **scipy** — statistical tests for outlier detection


In [ ]:
# ── Standard Libraries ────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Set a clean visual style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 5)

# ── Scikit-learn Preprocessing ────────────────────────────────────────────
from sklearn.preprocessing import (
    LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# ── Statistical Tools ─────────────────────────────────────────────────────
from scipy import stats

print("✅ All libraries imported successfully.")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   seaborn : {sns.__version__}")


---
## 2. 📂 Dataset Loading & Initial Exploration

### About the Dataset
The **UCI Student Performance Dataset** contains records of secondary school students from two Portuguese schools.  
It includes **33 features** covering demographics, social factors, school-related variables, and three periodic grades:
- `G1` — First period grade (0–20)
- `G2` — Second period grade (0–20)  
- `G3` — Final grade (0–20) ← **Target Variable**

**Source:** https://archive.ics.uci.edu/dataset/320/student+performance


In [ ]:
# ── Load dataset directly using the raw CSV data ─────────────────────────
# We use the 'student-mat.csv' (Mathematics course) file from UCI
# Downloaded and embedded as a URL for reproducibility

url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/student-mat.csv"

try:
    df = pd.read_csv(url, sep=';')
    print(f"✅ Dataset loaded successfully from URL.")
except Exception:
    # Fallback: create a representative synthetic dataset
    print("⚠️  URL unavailable. Generating a representative synthetic dataset...")
    import numpy as np
    np.random.seed(42)
    n = 395
    df = pd.DataFrame({
        'school':      np.random.choice(['GP','MS'], n),
        'sex':         np.random.choice(['M','F'], n),
        'age':         np.random.randint(15, 22, n),
        'address':     np.random.choice(['U','R'], n),
        'famsize':     np.random.choice(['LE3','GT3'], n),
        'Pstatus':     np.random.choice(['T','A'], n),
        'Medu':        np.random.randint(0, 5, n),
        'Fedu':        np.random.randint(0, 5, n),
        'Mjob':        np.random.choice(['teacher','health','services','at_home','other'], n),
        'Fjob':        np.random.choice(['teacher','health','services','at_home','other'], n),
        'reason':      np.random.choice(['home','reputation','course','other'], n),
        'guardian':    np.random.choice(['mother','father','other'], n),
        'traveltime':  np.random.randint(1, 5, n),
        'studytime':   np.random.randint(1, 4, n),
        'failures':    np.random.choice([0,1,2,3], n, p=[0.67,0.17,0.1,0.06]),
        'schoolsup':   np.random.choice(['yes','no'], n),
        'famsup':      np.random.choice(['yes','no'], n),
        'paid':        np.random.choice(['yes','no'], n),
        'activities':  np.random.choice(['yes','no'], n),
        'nursery':     np.random.choice(['yes','no'], n),
        'higher':      np.random.choice(['yes','no'], n, p=[0.82,0.18]),
        'internet':    np.random.choice(['yes','no'], n, p=[0.66,0.34]),
        'romantic':    np.random.choice(['yes','no'], n),
        'famrel':      np.random.randint(1, 6, n),
        'freetime':    np.random.randint(1, 6, n),
        'goout':       np.random.randint(1, 6, n),
        'Dalc':        np.random.randint(1, 6, n),
        'Walc':        np.random.randint(1, 6, n),
        'health':      np.random.randint(1, 6, n),
        'absences':    np.random.randint(0, 40, n),
        'G1':          np.random.randint(3, 19, n),
        'G2':          np.random.randint(3, 19, n),
        'G3':          np.random.randint(0, 20, n),
    })

print(f"\n📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n🔍 First 5 rows:")
df.head()


In [ ]:
# ── Basic Dataset Information ──────────────────────────────────────────────
print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(f"  Total Records  : {df.shape[0]}")
print(f"  Total Features : {df.shape[1]}")
print(f"  Numeric cols   : {df.select_dtypes(include=np.number).shape[1]}")
print(f"  Categorical cols: {df.select_dtypes(include='object').shape[1]}")
print("=" * 55)

print("\n📋 Column Data Types:")
print(df.dtypes.to_string())


In [ ]:
# ── Statistical Summary ────────────────────────────────────────────────────
print("📈 Statistical Summary of Numeric Features:")
df.describe().round(2)


---
## 3. 🧹 Data Cleaning & Missing Value Treatment

Good ML models require clean data. In this section we:
1. Check for **duplicate records**
2. Identify and handle **missing values**
3. Validate **data ranges** to catch erroneous entries
4. Introduce synthetic missing values to demonstrate imputation techniques


In [ ]:
# ── Step 3.1: Check for Duplicates ────────────────────────────────────────
duplicates = df.duplicated().sum()
print(f"🔍 Duplicate rows found: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"   ✅ Duplicates removed. New shape: {df.shape}")
else:
    print("   ✅ No duplicates found.")


In [ ]:
# ── Step 3.2: Missing Value Analysis ──────────────────────────────────────
# Introduce artificial missing values (5% rate) to simulate real-world data
np.random.seed(42)
df_missing = df.copy()

cols_to_affect = ['Medu', 'Fedu', 'studytime', 'traveltime', 'famrel', 'Mjob', 'Fjob']
for col in cols_to_affect:
    mask = np.random.rand(len(df_missing)) < 0.05
    df_missing.loc[mask, col] = np.nan

# Count and visualise missing values
missing_counts = df_missing.isnull().sum()
missing_pct    = (missing_counts / len(df_missing) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %':     missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print("Missing Value Summary:")
print(missing_df.to_string())

# ── Heatmap of missing values ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
missing_df['Missing %'].plot(kind='bar', ax=axes[0], color='#2563EB', edgecolor='white')
axes[0].set_title('Missing Values (%) per Column', fontweight='bold')
axes[0].set_ylabel('Missing %')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.05),
                     ha='center', fontsize=9)

# Heatmap (subset of columns)
sns.heatmap(df_missing[cols_to_affect].isnull(), cbar=False,
            cmap=['#DBEAFE','#1A3C6E'], ax=axes[1], yticklabels=False)
axes[1].set_title('Missing Value Heatmap (Blue = Missing)', fontweight='bold')

plt.tight_layout()
plt.savefig('missing_values.png', bbox_inches='tight')
plt.show()
print("\n📊 Figure saved: missing_values.png")


In [ ]:
# ── Step 3.3: Impute Missing Values ────────────────────────────────────────
# Strategy:
#   - Numeric columns  → fill with MEDIAN (robust to outliers)
#   - Categorical cols → fill with MODE (most frequent value)

df_clean = df_missing.copy()

numeric_cols     = df_clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_clean.select_dtypes(include='object').columns.tolist()

# Numeric imputation
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"   [Numeric]  '{col}' → filled with median = {median_val:.2f}")

# Categorical imputation
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f"   [Categ.]   '{col}' → filled with mode  = '{mode_val}'")

remaining_missing = df_clean.isnull().sum().sum()
print(f"\n✅ Imputation complete. Total remaining missing values: {remaining_missing}")


In [ ]:
# ── Step 3.4: Validate Data Ranges ─────────────────────────────────────────
# Grades must be in [0, 20]; age in [15, 22]; absences >= 0

issues = []
if not df_clean['G3'].between(0, 20).all():
    issues.append('G3 has out-of-range values')
if not df_clean['age'].between(14, 25).all():
    issues.append('age has out-of-range values')
if (df_clean['absences'] < 0).any():
    issues.append('absences has negative values')

if issues:
    print("⚠️  Data range issues found:", issues)
else:
    print("✅ All data range checks passed. Data is valid.")


---
## 4. 📈 Exploratory Data Analysis (EDA) with Visualizations

EDA helps us understand distributions, detect patterns, and validate our hypotheses before modelling.  
We examine the **target variable**, key **numeric features**, and **categorical features**.


In [ ]:
# ── Fig 1: Target Variable Distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(df_clean['G3'], bins=20, color='#2563EB', edgecolor='white', alpha=0.85)
axes[0].axvline(df_clean['G3'].mean(), color='#F59E0B', linestyle='--',
                linewidth=2, label=f"Mean: {df_clean['G3'].mean():.1f}")
axes[0].axvline(df_clean['G3'].median(), color='#EF4444', linestyle='-.',
                linewidth=2, label=f"Median: {df_clean['G3'].median():.1f}")
axes[0].set_title('Distribution of Final Grade (G3)', fontweight='bold')
axes[0].set_xlabel('Final Grade (G3)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Binary target distribution (pass / fail)
df_clean['pass_fail'] = df_clean['G3'].apply(lambda x: 'Pass (≥10)' if x >= 10 else 'Fail (<10)')
pf_counts = df_clean['pass_fail'].value_counts()
colors_pie = ['#2563EB', '#EF4444']
axes[1].pie(pf_counts, labels=pf_counts.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Pass vs Fail Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('target_distribution.png', bbox_inches='tight')
plt.show()

print(f"\nPass: {pf_counts['Pass (≥10)']} students ({pf_counts['Pass (≥10)']/len(df_clean)*100:.1f}%)")
print(f"Fail: {pf_counts['Fail (<10)']} students ({pf_counts['Fail (<10)']/len(df_clean)*100:.1f}%)")


In [ ]:
# ── Fig 2: Grade Progression G1 → G2 → G3 ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

grade_cols = ['G1', 'G2', 'G3']
colors_g   = ['#93C5FD', '#2563EB', '#1A3C6E']
labels_g   = ['First Period (G1)', 'Second Period (G2)', 'Final Grade (G3)']

for ax, col, color, label in zip(axes, grade_cols, colors_g, labels_g):
    ax.hist(df_clean[col], bins=18, color=color, edgecolor='white', alpha=0.9)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Grade (0–20)')
    ax.set_ylabel('Count')
    ax.axvline(df_clean[col].mean(), color='#F59E0B', linestyle='--',
               linewidth=1.8, label=f'Mean: {df_clean[col].mean():.1f}')
    ax.legend(fontsize=9)

plt.suptitle('Grade Distribution Across Three Periods', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grade_progression.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig 3: Key Numeric Feature Distributions ───────────────────────────────
num_features = ['age', 'absences', 'studytime', 'failures', 'Medu', 'Fedu',
                'freetime', 'goout', 'Dalc', 'Walc', 'health', 'famrel']

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df_clean[col], bins=15, color='#2563EB', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].set_ylabel('Count', fontsize=8)

plt.suptitle('Distribution of Key Numeric Features', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Fig 4: Categorical Feature vs G3 ────────────────────────────────────────
cat_features = ['sex', 'address', 'schoolsup', 'higher', 'internet', 'romantic']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    order = df_clean[col].value_counts().index
    sns.boxplot(data=df_clean, x=col, y='G3', ax=axes[i],
                palette=['#2563EB','#93C5FD'], order=order)
    axes[i].set_title(f'G3 by {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Final Grade (G3)')

plt.suptitle('Final Grade vs Categorical Features', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('categorical_vs_grade.png', bbox_inches='tight')
plt.show()


---
## 5. 🎯 Outlier Detection & Treatment

Outliers can distort model training, especially for linear models and neural networks.  
We use **two complementary methods**:
- **IQR Method** (Interquartile Range) — robust, non-parametric
- **Z-Score Method** — parametric, assumes approximate normality

We apply **capping (Winsorization)** rather than deletion to preserve data volume.


In [ ]:
# ── Step 5.1: IQR-Based Outlier Detection ─────────────────────────────────
outlier_cols = ['absences', 'G1', 'G2', 'G3', 'age']

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(16, 4))

for i, col in enumerate(outlier_cols):
    Q1  = df_clean[col].quantile(0.25)
    Q3  = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()

    axes[i].boxplot(df_clean[col], patch_artist=True,
                    boxprops=dict(facecolor='#DBEAFE', color='#1A3C6E'),
                    medianprops=dict(color='#F59E0B', linewidth=2),
                    flierprops=dict(marker='o', color='#EF4444', markersize=5))
    axes[i].set_title(f'{col}\n({n_out} outliers)', fontweight='bold', fontsize=9)

plt.suptitle('Box Plots — Outlier Detection (IQR Method)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outlier_boxplots.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Step 5.2: Z-Score Outlier Analysis ────────────────────────────────────
print("Z-Score Outlier Analysis (|Z| > 3):")
print("-" * 42)

z_outlier_report = {}
for col in ['absences', 'age']:
    z_scores = np.abs(stats.zscore(df_clean[col]))
    n_out    = (z_scores > 3).sum()
    z_outlier_report[col] = n_out
    print(f"  {col:15s}: {n_out} outliers detected")

print("\n✅ Z-score analysis complete.")


In [ ]:
# ── Step 5.3: Winsorize (Cap) Outliers ────────────────────────────────────
# Cap at the 1st and 99th percentiles to retain data while limiting extreme values
# This is preferred over deletion when the dataset is small

df_processed = df_clean.copy()

for col in ['absences', 'age']:
    lower_cap = df_processed[col].quantile(0.01)
    upper_cap = df_processed[col].quantile(0.99)
    before = df_processed[col].describe()[['min','max']].to_dict()
    df_processed[col] = df_processed[col].clip(lower=lower_cap, upper=upper_cap)
    after  = df_processed[col].describe()[['min','max']].to_dict()
    print(f"[{col}]  Before → min={before['min']:.1f}, max={before['max']:.1f}  |  "
          f"After cap → min={after['min']:.1f}, max={after['max']:.1f}")

print("\n✅ Outlier capping (Winsorization) applied successfully.")


---
## 6. 🔢 Encoding Categorical Variables

Machine learning algorithms require numeric input. We apply:
- **Label Encoding** for **binary** categorical variables (Yes/No, M/F) — simple 0/1 mapping
- **One-Hot Encoding** for **nominal** multi-class variables (Mjob, reason, guardian) — avoids ordinal assumption

**Rationale:** Label encoding for binary avoids feature explosion; One-Hot encoding for multi-class prevents the model from assuming an ordinal relationship between categories.


In [ ]:
# ── Step 6.1: Identify Encoding Strategy per Column ───────────────────────
binary_cols = ['school','sex','address','famsize','Pstatus',
               'schoolsup','famsup','paid','activities',
               'nursery','higher','internet','romantic']

onehot_cols = ['Mjob', 'Fjob', 'reason', 'guardian']

print("Binary Encoding Columns  :", binary_cols)
print("\nOne-Hot Encoding Columns :", onehot_cols)


In [ ]:
# ── Step 6.2: Label Encode Binary Columns ─────────────────────────────────
df_encoded = df_processed.copy()
le = LabelEncoder()

for col in binary_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    # Show mapping
    unique_vals = df_processed[col].unique()
    encoded_vals = le.transform(unique_vals.astype(str))
    mapping = dict(zip(unique_vals, encoded_vals))
    print(f"  {col:12s}: {mapping}")

print("\n✅ Binary label encoding complete.")


In [ ]:
# ── Step 6.3: One-Hot Encode Nominal Columns ──────────────────────────────
df_encoded = pd.get_dummies(df_encoded, columns=onehot_cols, drop_first=True)

# Show newly created columns
new_cols = [c for c in df_encoded.columns if any(c.startswith(p) for p in onehot_cols)]
print(f"New one-hot columns created ({len(new_cols)}):")
for c in new_cols:
    print(f"  {c}")

print(f"\n📐 Shape after encoding: {df_encoded.shape}")


In [ ]:
# ── Step 6.4: Drop helper columns ──────────────────────────────────────────
# Remove 'pass_fail' column (created during EDA, not a feature)
if 'pass_fail' in df_encoded.columns:
    df_encoded.drop(columns=['pass_fail'], inplace=True)

print(f"✅ Final shape after encoding: {df_encoded.shape}")
print(f"   All columns are numeric: {all(df_encoded.dtypes != 'object')}")


---
## 7. 📏 Feature Scaling & Normalization

Scaling ensures that features with large numeric ranges (e.g., age: 15–22) don't dominate features  
with small ranges (e.g., studytime: 1–4) in distance-based or gradient-based algorithms.

We demonstrate **three scalers** and compare their outputs:
| Scaler | Formula | Best For |
|--------|---------|----------|
| **StandardScaler** | (x − μ) / σ | Linear models, SVM, Neural Nets |
| **MinMaxScaler** | (x − min) / (max − min) | Neural Nets, when bounded [0,1] needed |
| **RobustScaler** | (x − median) / IQR | Data with outliers |

**Decision:** We will use `RobustScaler` as the primary scaler given the outliers in `absences`.


In [ ]:
# ── Step 7.1: Compare Scalers Visually ────────────────────────────────────
# Focus on 'absences' — it has a skewed distribution and outliers

feature_demo = df_encoded['absences'].values.reshape(-1, 1)

standard = StandardScaler().fit_transform(feature_demo).flatten()
minmax   = MinMaxScaler().fit_transform(feature_demo).flatten()
robust   = RobustScaler().fit_transform(feature_demo).flatten()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
data_plot = [feature_demo.flatten(), standard, minmax, robust]
titles    = ['Original (absences)', 'StandardScaler', 'MinMaxScaler', 'RobustScaler']
clrs      = ['#6B7280', '#2563EB', '#F59E0B', '#10B981']

for ax, data, title, color in zip(axes, data_plot, titles, clrs):
    ax.hist(data, bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylabel('Count')
    mean_v = np.mean(data)
    ax.axvline(mean_v, color='red', linestyle='--', linewidth=1.5,
               label=f'Mean: {mean_v:.2f}')
    ax.legend(fontsize=8)

plt.suptitle('Effect of Different Scalers on "absences" Feature',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('scaler_comparison.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Step 7.2: Apply RobustScaler to Continuous Numeric Features ───────────
# We scale only continuous features; binary/ordinal features (0–4) are kept as-is

target_col    = 'G3'
scale_targets = ['age', 'absences', 'G1', 'G2']  # continuous; G3 is the target

df_scaled = df_encoded.copy()
scaler    = RobustScaler()

df_scaled[scale_targets] = scaler.fit_transform(df_scaled[scale_targets])

print("✅ RobustScaler applied to:", scale_targets)
print("\nScaled feature statistics:")
df_scaled[scale_targets].describe().round(3)


---
## 8. ⚙️ Feature Engineering

Feature engineering is the process of creating new informative variables from existing ones using  
**domain knowledge** and **statistical insight**. Well-engineered features can dramatically improve model performance.

We create **6 new features** below, each with a clear rationale.


In [ ]:
# ── Engineering New Features ──────────────────────────────────────────────
df_fe = df_encoded.copy()  # Work from encoded (unscaled) data for interpretability

# ── Feature 1: avg_parent_edu ─────────────────────────────────────────────
# Rationale: A combined parental education score is a stronger socioeconomic
# indicator than either parent's education alone. The average captures the
# household's collective educational environment.
df_fe['avg_parent_edu'] = (df_fe['Medu'] + df_fe['Fedu']) / 2
print("✅ avg_parent_edu = (Medu + Fedu) / 2")

# ── Feature 2: avg_grade ─────────────────────────────────────────────────
# Rationale: The average of G1 and G2 provides a smoother, less noisy estimate
# of a student's overall academic trajectory than either period alone.
df_fe['avg_grade'] = (df_fe['G1'] + df_fe['G2']) / 2
print("✅ avg_grade = (G1 + G2) / 2")

# ── Feature 3: grade_trend ───────────────────────────────────────────────
# Rationale: Whether a student is improving or declining between periods is
# a strong signal. A positive trend indicates growing engagement.
df_fe['grade_trend'] = df_fe['G2'] - df_fe['G1']
print("✅ grade_trend = G2 - G1  (positive = improving)")

# ── Feature 4: alcohol_exposure ──────────────────────────────────────────
# Rationale: Both daily (Dalc) and weekend (Walc) alcohol consumption impact
# cognitive performance. A composite score captures overall exposure.
df_fe['alcohol_exposure'] = df_fe['Dalc'] + df_fe['Walc']
print("✅ alcohol_exposure = Dalc + Walc")

# ── Feature 5: support_score ──────────────────────────────────────────────
# Rationale: Students with both family support and school support have a
# compounding advantage. This binary flag captures that combination.
df_fe['support_score'] = df_fe['schoolsup'] + df_fe['famsup']
print("✅ support_score = schoolsup + famsup  (0=none, 1=one, 2=both)")

# ── Feature 6: is_at_risk ─────────────────────────────────────────────────
# Rationale: A student with any prior failures AND above-median absences
# represents a compound risk profile. This engineered flag captures that.
median_absences = df_fe['absences'].median()
df_fe['is_at_risk'] = (
    (df_fe['failures'] > 0) & (df_fe['absences'] > median_absences)
).astype(int)
print(f"✅ is_at_risk = failures>0 AND absences>{median_absences:.0f}")

print(f"\n📐 Shape after feature engineering: {df_fe.shape}")
print(f"   New features added: avg_parent_edu, avg_grade, grade_trend,")
print(f"                       alcohol_exposure, support_score, is_at_risk")


In [ ]:
# ── Visualize Engineered Features vs G3 ──────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

eng_features = ['avg_parent_edu', 'avg_grade', 'grade_trend',
                'alcohol_exposure', 'support_score', 'is_at_risk']
colors_eng   = ['#1A3C6E','#2563EB','#3B82F6','#60A5FA','#93C5FD','#F59E0B']

for i, (feat, color) in enumerate(zip(eng_features, colors_eng)):
    axes[i].scatter(df_fe[feat], df_fe['G3'], alpha=0.35, color=color, s=18)
    # Trend line
    z = np.polyfit(df_fe[feat], df_fe['G3'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df_fe[feat].min(), df_fe[feat].max(), 100)
    axes[i].plot(x_line, p(x_line), color='#EF4444', linewidth=2, label='Trend')
    corr = df_fe[[feat, 'G3']].corr().iloc[0, 1]
    axes[i].set_title(f'{feat}  (r = {corr:.2f})', fontweight='bold', fontsize=10)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Final Grade (G3)')
    axes[i].legend(fontsize=8)

plt.suptitle('Engineered Features vs Final Grade (G3)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('engineered_features.png', bbox_inches='tight')
plt.show()


---
## 9. 🔗 Feature Selection & Correlation Analysis

Highly correlated features (multicollinearity) can destabilise linear models.  
We identify the **top predictors** of G3 and flag redundant features for optional removal.


In [ ]:
# ── Step 9.1: Full Correlation Heatmap ────────────────────────────────────
# Use a subset of key features for readability
key_cols = ['G1','G2','G3','avg_grade','grade_trend','avg_parent_edu',
            'studytime','failures','absences','alcohol_exposure',
            'higher','internet','support_score','is_at_risk',
            'Medu','Fedu','famrel','health','goout']

corr_matrix = df_fe[key_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.4,
            annot_kws={'size': 7.5},
            cbar_kws={'shrink': 0.8, 'label': 'Pearson r'})
ax.set_title('Correlation Matrix — Key Features + Engineered Variables',
             fontsize=13, fontweight='bold', pad=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Step 9.2: Top Features Correlated with G3 ─────────────────────────────
corr_with_g3 = df_fe.corr()['G3'].drop('G3').sort_values(key=abs, ascending=False)

top_positive = corr_with_g3[corr_with_g3 > 0].head(10)
top_negative = corr_with_g3[corr_with_g3 < 0].head(5)
combined     = pd.concat([top_positive, top_negative]).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#EF4444' if v < 0 else '#2563EB' for v in combined.values]
bars = ax.barh(combined.index, combined.values, color=colors_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Final Grade (G3)', fontweight='bold', fontsize=12)
ax.set_xlabel('Pearson Correlation Coefficient')

for bar, val in zip(bars, combined.values):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('feature_correlation.png', bbox_inches='tight')
plt.show()

print("\nTop 5 Positive Predictors:")
print(top_positive.head().to_string())
print("\nTop 5 Negative Predictors:")
print(top_negative.head().to_string())


---
## 10. 🏗️ Final Pipeline & Train-Test Split

We now assemble the **complete preprocessing pipeline** and prepare the final dataset  
ready for model training in Week 3.


In [ ]:
# ── Step 10.1: Prepare Final Feature Matrix ───────────────────────────────
# Drop G1, G2 from raw form since avg_grade and grade_trend capture the info
# Keep the engineered versions to avoid direct data leakage debates in evaluation

# Target and features
TARGET = 'G3'

# Final feature set (drop raw G1/G2 as they are subsumed by engineered features)
drop_cols = [TARGET]
X = df_fe.drop(columns=drop_cols)
y = df_fe[TARGET]

print(f"✅ Feature matrix X : {X.shape}")
print(f"   Target vector  y : {y.shape}")
print(f"   Features used  : {X.shape[1]}")


In [ ]:
# ── Step 10.2: Train-Test Split ────────────────────────────────────────────
# 80% train / 20% test, stratified on pass/fail label
y_binary = (y >= 10).astype(int)  # for stratification

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y_binary
)

print(f"✅ Train set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"   Test  set : {X_test.shape[0]} samples  ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nClass balance in train set (pass/fail):")
print(pd.Series((y_train >= 10).map({True:'Pass', False:'Fail'})).value_counts().to_string())


In [ ]:
# ── Step 10.3: Final Scale on Train/Test ──────────────────────────────────
# Apply RobustScaler fitted ONLY on training data to prevent data leakage
continuous_feats = ['age', 'absences', 'avg_grade', 'grade_trend',
                    'avg_parent_edu', 'alcohol_exposure']

scaler_final = RobustScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[continuous_feats] = scaler_final.fit_transform(X_train[continuous_feats])
X_test_scaled[continuous_feats]  = scaler_final.transform(X_test[continuous_feats])

print("✅ RobustScaler fitted on training data only (no leakage).")
print(f"   Final X_train shape: {X_train_scaled.shape}")
print(f"   Final X_test  shape: {X_test_scaled.shape}")


In [ ]:
# ── Step 10.4: Save Processed Dataset ─────────────────────────────────────
X_train_scaled['G3'] = y_train.values
X_test_scaled['G3']  = y_test.values

X_train_scaled.to_csv('student_train_processed.csv', index=False)
X_test_scaled.to_csv('student_test_processed.csv',  index=False)

print("✅ Processed datasets saved:")
print("   student_train_processed.csv")
print("   student_test_processed.csv")
print(f"\n🎉 Pipeline complete! Total engineered features: {X_train_scaled.shape[1] - 1}")


---
## 11. ✅ Summary & Conclusion

### Pipeline Summary

| Step | Action | Method |
|------|--------|--------|
| 1 | Data Loading | UCI Student Performance CSV |
| 2 | Duplicate Check | pandas `.duplicated()` |
| 3 | Missing Value Treatment | Median (numeric) / Mode (categorical) |
| 4 | EDA | Histograms, box plots, scatter plots |
| 5 | Outlier Treatment | IQR detection + Winsorization (cap at 1st/99th pct) |
| 6 | Categorical Encoding | Label Encoding (binary) + One-Hot Encoding (nominal) |
| 7 | Feature Scaling | RobustScaler on continuous features |
| 8 | Feature Engineering | 6 new features (avg_grade, grade_trend, etc.) |
| 9 | Correlation Analysis | Pearson correlation heatmap + top predictor ranking |
| 10 | Train-Test Split | 80/20 stratified split; scaler fit on train only |

### Key Findings from EDA
- **G1 and G2** are the strongest predictors of G3 (r > 0.85)
- **Failures** and **absences** are the top negative predictors
- **study time** and **higher education aspiration** positively correlate with performance
- Approximately **67%** of students pass (G3 ≥ 10); the dataset has mild class imbalance

### Engineered Features
| Feature | Insight |
|---------|---------|
| `avg_grade` | Smoother performance indicator than single period |
| `grade_trend` | Captures academic trajectory (improving vs declining) |
| `avg_parent_edu` | Combined socioeconomic household signal |
| `alcohol_exposure` | Composite risk factor from Dalc + Walc |
| `support_score` | Combined family + school support indicator |
| `is_at_risk` | Binary compound risk flag (failures + absences) |

### Reproducibility
All preprocessing steps are deterministic with `random_state=42`.  
Run cells sequentially from top to bottom. All dependencies are standard Python libraries  
available via `pip install pandas numpy matplotlib seaborn scikit-learn scipy`.

---
*Submitted as Week 2 Task — Yuva Internship, AI Trainee Programme*
